# Website Insights Agent — Kaggle Capstone Demo

This notebook demonstrates the Website Insights Agent end-to-end using sample data.
It serves as the submission artifact for the AI Agents: Intensive Vibe Coding Course.

**Agent flow:**
1. Clone the project repo and install dependencies
2. Load GA4 + Search Console data (sample CSVs here; S3 in production)
3. Summarize data into a structured prompt context
4. Call Gemini to generate actionable recommendations
5. Format and display recommendations (email step skipped in notebook mode)

**Course concepts demonstrated:**
- Agent loop with tool orchestration
- Multiple specialized tools as plain Python functions
- Structured prompt context building
- Config-driven design for portability (Kaggle → AWS Bedrock)


## Setup — Clone Repo and Install Dependencies

In [ ]:
!git clone https://github.com/KazaFamily/website-insights-agent.git

In [ ]:
!pip install -r website-insights-agent/requirements.txt -q

In [ ]:
import sys
sys.path.append('website-insights-agent')

## Step 1: Configure Gemini API Key

Add your Gemini API key via Kaggle Secrets (Add-ons → Secrets → GEMINI_API_KEY).

In [ ]:
import os
from google import genai
from google.genai import types
# Kaggle Secrets — uncomment when running on Kaggle
from kaggle_secrets import UserSecretsClient
os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')

# Local dev fallback
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
print('Gemini configured.')

## Step 2: Load Sample Data

In production these DataFrames are loaded from S3 via `agent/tools/s3_reader.py`.
Here we load the same file formats directly from `data/sample/`.

Note: GA4 exports include a 9-line metadata header — we skip it with `skiprows=9`.

In [ ]:
import pandas as pd

DATA_DIR = '/kaggle/working/website-insights-agent/data/sample'

# GA4 exports — skip 9-line metadata header
ga4_pages_df = pd.read_csv(
    f'{DATA_DIR}/Pages_and_screens_Page_title_and_screen_class_90_days.csv',
    skiprows=9
)

ga4_traffic_df = pd.read_csv(
    f'{DATA_DIR}/Traffic_acquisition_Session_source_medium_90_days.csv',
    skiprows=9
)

# Search Console exports — clean CSVs, no header offset
sc_pages_df = pd.read_csv(f'{DATA_DIR}/search_console_pages_3_months.csv')
sc_queries_df = pd.read_csv(f'{DATA_DIR}/search_console_queries_3_months.csv')

print(f'GA4 Pages: {len(ga4_pages_df)} rows')
print(f'GA4 Traffic: {len(ga4_traffic_df)} rows')
print(f'Search Console Pages: {len(sc_pages_df)} rows')
print(f'Search Console Queries: {len(sc_queries_df)} rows')

## Step 3: Summarize Data (Analyzer Tool)

The analyzer tool converts raw DataFrames into a structured text context
that the agent passes to Gemini. This is the core data preparation step.

In [ ]:
from agent.tools.analyzer import build_analysis_context

SITE_URL = 'https://sri-kaza.com'

context = build_analysis_context(
    ga4_pages_df=ga4_pages_df,
    ga4_traffic_df=ga4_traffic_df,
    sc_pages_df=sc_pages_df,
    sc_queries_df=sc_queries_df,
    site_url=SITE_URL,
)

print(context)

## Step 4: Agent Loop — Call Gemini

The agent passes the structured context to Gemini with a focused system prompt.
Gemini acts as the reasoning layer — identifying patterns and generating
prioritized, actionable recommendations for the site owner.

In [ ]:
SYSTEM_PROMPT = """You are a website performance analyst for a small business site.
You will receive 90-day data from Google Analytics 4 (GA4) and Google Search Console.

Your job is to identify the top 5 actionable recommendations, prioritized by impact.

For each recommendation:
- Name the specific page or query you are referring to
- Explain the problem or opportunity in plain terms
- Suggest one concrete action the site owner can take this week

Rules:
- Write for a non-technical audience
- Be specific — avoid generic SEO advice
- Prioritize quick wins (high impression / low CTR, near-top rankings) over long-term projects
- Keep the total response under 600 words
"""

user_prompt = (
    f"Here is the latest 90-day website data for {SITE_URL}:\n\n"
    f"{context}\n\n"
    f"Please provide your top 5 recommendations."
)

response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.2,
        max_output_tokens=2048,
    ),
)

recommendations = response.text
print(recommendations)

## Step 5: Format Output

In production, `agent/tools/email_sender.py` POSTs these recommendations
to an AWS Lambda function URL, which triggers SES to deliver them via email
on a daily or weekly schedule.

Here we display the formatted email body directly.

In [ ]:
from agent.tools.email_sender import format_email_body

email_body = format_email_body(SITE_URL, recommendations)
print(email_body)

## Production Architecture

```
AWS Lambda (scheduled trigger)
        │
        ▼
  agent/tools/s3_reader.py   ← loads GA4 + Search Console CSVs from S3
        │
        ▼
  agent/tools/analyzer.py    ← summarizes data into structured prompt context
        │
        ▼
  Gemini API (raw call)      ← generates prioritized recommendations
        │
        ▼
  agent/tools/email_sender.py ← POSTs to Lambda → SES → your inbox
```

**Designed for portability:** raw API calls (no ADK/LangGraph) make this
straightforward to migrate to AWS Bedrock by swapping the Gemini client
for a Bedrock Claude client with no structural changes.

**GitHub:** https://github.com/KazaFamily/website-insights-agent
